# Laboratorio: operadores, adjunto y subespacios invariantes

En este laboratorio usaremos cálculo exacto para reconocer subespacios invariantes, obtener restricciones, calcular adjuntos y verificar sus relaciones con núcleo, imagen y proyecciones ortogonales.

## 0. Preparación

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing()
np.set_printoptions(precision=5, suppress=True)

## 1. Cómo comprobar la invariancia

Si las columnas de $B_W$ forman una base de $W$, entonces $W$ es invariante bajo $A$ si y solo si cada columna de $AB_W$ es combinación lineal de las columnas de $B_W$. Equivalentemente, $\operatorname{rango}[B_W\;AB_W]=\operatorname{rango}(B_W)$.

In [ ]:
def datos_invariancia(A, BW):
    A, BW = sp.Matrix(A), sp.Matrix(BW)
    es_invariante = BW.row_join(A * BW).rank() == BW.rank()
    restriccion = None
    if es_invariante:
        columnas = [BW.gauss_jordan_solve(A * BW.col(j))[0]
                    for j in range(BW.cols)]
        restriccion = sp.Matrix.hstack(*columnas)
    return es_invariante, restriccion

A = sp.Matrix([[2, 1], [0, 3]])
BW = sp.Matrix([[1], [0]])
inv_W, A_W = datos_invariancia(A, BW)
print("¿W = span(e1) es invariante bajo A?", inv_W)
print("Matriz de la restricción:")
sp.pprint(A_W)
assert inv_W and A_W == sp.Matrix([[2]])

El complemento ortogonal de $W$ es $W^\perp=\operatorname{span}\{e_2\}$. Comprobamos que no es invariante bajo $A$, pero sí bajo el adjunto $A^T$.

In [ ]:
BW_perp = sp.Matrix([[0], [1]])
inv_perp_A, _ = datos_invariancia(A, BW_perp)
inv_perp_adj, restr_perp_adj = datos_invariancia(A.T, BW_perp)
print("¿W_perp es invariante bajo A?", inv_perp_A)
print("¿W_perp es invariante bajo A.T?", inv_perp_adj)
assert not inv_perp_A
assert inv_perp_adj and restr_perp_adj == sp.Matrix([[3]])

### Visualización de la invariancia

La recta horizontal $W$ se conserva bajo $A$. La recta vertical $W^\perp$ no se conserva bajo $A$, porque la imagen de $e_2$ adquiere una componente horizontal.

In [ ]:
A_np = np.array(A, dtype=float)
e1, e2 = np.eye(2)
fig, ax = plt.subplots(figsize=(6, 6))
ax.axhline(0, color="#2a6fbb", lw=2, label=r"$W=\mathrm{span}(e_1)$")
ax.axvline(0, color="#999999", lw=2, label=r"$W^\perp=\mathrm{span}(e_2)$")
for v, color, label in [(e1, "#1b9e77", r"$e_1$"),
                         (A_np @ e1, "#0b6e4f", r"$Ae_1$"),
                         (e2, "#d95f02", r"$e_2$"),
                         (A_np @ e2, "#a33b00", r"$Ae_2$")]:
    ax.quiver(0, 0, v[0], v[1], angles="xy", scale_units="xy",
              scale=1, color=color, width=0.012, label=label)
ax.set(xlim=(-0.5, 3.7), ylim=(-0.5, 3.7), aspect="equal",
       xlabel="primera coordenada", ylabel="segunda coordenada")
ax.grid(alpha=0.25)
ax.legend(loc="upper left", ncol=2)
plt.show()

## 2. Forma por bloques y matriz de la restricción

En el siguiente ejemplo, las dos primeras columnas generan un subespacio invariante. La base ya está adaptada a ese subespacio, de modo que aparece un bloque inferior izquierdo nulo.

In [ ]:
C = sp.Matrix([[1, 2, -1], [0, 3, 4], [0, 0, 5]])
BW2 = sp.eye(3)[:, :2]
inv_W2, C_W = datos_invariancia(C, BW2)
print("¿span(e1,e2) es invariante?", inv_W2)
print("Matriz de la restricción:")
sp.pprint(C_W)
assert inv_W2
assert C_W == C[:2, :2]
assert C[2, :2] == sp.zeros(1, 2)

## 3. Adjunto en una base ortonormal

Con el producto interno usual, la matriz del adjunto en la base canónica es la transpuesta en el caso real y la transpuesta conjugada en el caso complejo. Usaremos la convención $\langle x,y\rangle=y^*x$.

In [ ]:
x = sp.Matrix([2, -1])
y = sp.Matrix([3, 4])
lado_izq = (y.T * A * x)[0]
lado_der = ((A.T * y).T * x)[0]
print("<Ax,y> =", lado_izq)
print("<x,A.T y> =", lado_der)
assert sp.simplify(lado_izq - lado_der) == 0

In [ ]:
def producto_complejo(u, v):
    return (sp.conjugate(v).T * u)[0]

Ac = sp.Matrix([[1, sp.I], [2, 1-sp.I]])
Ac_adj = sp.conjugate(Ac).T
xc = sp.Matrix([1+sp.I, 2])
yc = sp.Matrix([3, 1-sp.I])
print("Adjunto complejo:")
sp.pprint(Ac_adj)
assert sp.simplify(producto_complejo(Ac*xc, yc)
                   - producto_complejo(xc, Ac_adj*yc)) == 0

## 4. Adjunto con un producto interno ponderado

Si $\langle x,y\rangle_G=y^*Gx$, la matriz correcta del adjunto es $G^{-1}A^*G$. Este cálculo muestra por qué no basta transponer cuando la base no es ortonormal para el producto interno considerado.

In [ ]:
G = sp.diag(1, 2)
M = sp.Matrix([[0, 1], [1, 0]])
M_adj_G = G.inv() * M.T * G
print("Adjunto respecto del producto interno ponderado:")
sp.pprint(M_adj_G)

a, b, c, d = sp.symbols('a b c d', real=True)
u, v = sp.Matrix([a, b]), sp.Matrix([c, d])
producto_G = lambda p, q: (q.T * G * p)[0]
assert sp.simplify(producto_G(M*u, v) - producto_G(u, M_adj_G*v)) == 0
assert M_adj_G != M.T

## 5. Relaciones entre imagen y núcleo

Para una transformación rectangular también existe un adjunto. En bases ortonormales, si $T$ está representada por $R$, entonces $T^*$ está representada por $R^T$ en el caso real. Verificaremos $({\rm Im}\,T)^\perp=\ker T^*$ y $(\ker T)^\perp={\rm Im}\,T^*$.

In [ ]:
R = sp.Matrix([[1, 2, 3], [2, 4, 6]])
imagen_T = R.columnspace()
nucleo_T_adj = R.T.nullspace()
nucleo_T = R.nullspace()
imagen_T_adj = R.T.columnspace()

print("Base de Im(T):", imagen_T)
print("Base de ker(T*):", nucleo_T_adj)
print("Base de ker(T):", nucleo_T)
print("Base de Im(T*):", imagen_T_adj)

assert all((z.T*w)[0] == 0 for z in nucleo_T_adj for w in imagen_T)
assert all((z.T*w)[0] == 0 for z in nucleo_T for w in imagen_T_adj)
assert len(nucleo_T_adj) + len(imagen_T) == R.rows
assert len(nucleo_T) + len(imagen_T_adj) == R.cols

## 6. Proyección ortogonal

Si las columnas de $Q$ son ortonormales, $P=QQ^T$ es la proyección ortogonal sobre su espacio columna. Comprobaremos que es simétrica, idempotente y que anula el complemento ortogonal.

In [ ]:
q1 = sp.Matrix([1, 1, 0]) / sp.sqrt(2)
q2 = sp.Matrix([0, 0, 1])
Q = sp.Matrix.hstack(q1, q2)
P = sp.simplify(Q * Q.T)
z = sp.Matrix([1, -1, 0])
print("Matriz de la proyección:")
sp.pprint(P)
assert sp.simplify(P*P - P) == sp.zeros(3)
assert P.T == P
assert P*z == sp.zeros(3, 1)
assert P.rank() == 2

## 7. Actividades

1. Modifica la función `datos_invariancia` para devolver también las coordenadas de cada imagen en la base de $W$.
2. Construye una matriz de $3\times3$ para la cual $\operatorname{span}\{e_1,e_2\}$ sea invariante, pero su complemento ortogonal no lo sea. Comprueba qué ocurre con el adjunto.
3. Repite el cálculo ponderado con una matriz de Gram simétrica definida positiva no diagonal.
4. Genera una matriz $Q$ con dos columnas ortonormales en $\mathbb R^4$ y verifica las cuatro propiedades de la proyección ortogonal.

## 8. Cierre

- La invariancia se comprueba verificando que las imágenes de una base permanecen en el subespacio.
- Una base adaptada convierte la invariancia en ceros dentro de la matriz.
- El adjunto depende del producto interno y conecta núcleo, imagen y ortogonalidad.
- Una proyección ortogonal es idempotente y autoadjunta.